# แสดงข้อมูลพรรณไม้จากไฟล์ VRU.csv และส่งออกเป็น JSON
สมุดบันทึกนี้ใช้สำหรับเชื่อมต่อกับ Google Drive, โหลดเปิดแสดงผลข้อมูลพืชพรรณจากไฟล์ `data/VRU.csv` และดำเนินการแปลงพร้อมส่งออกไฟล์ข้อมูลเป็นรูปแบบ JSON เพื่อนำไปพัฒนาระบบต่อยอด

In [ ]:
import pandas as pd
import os
from google.colab import drive

# 1. เชื่อมต่อกับ Google Drive
try:
    drive.mount('/content/drive')
    print("✅ เชื่อมต่อ Google Drive สำเร็จ")
except Exception as e:
    print("⚠️ หากไม่ได้รันบน Google Colab สามารถข้ามการเชื่อมต่อไดรฟ์ได้:", e)

# 2. กำหนดพาธไฟล์ CSV บน Google Drive
csv_path = '/content/drive/MyDrive/Colab Notebooks/plants/data/VRU.csv'

# พาธสำรองเผื่อกรณีรันในเครื่องส่วนตัว (Local)
local_csv_path = 'data/VRU.csv'

# ตรวจสอบและเลือกพาธที่มีไฟล์อยู่จริง
selected_path = None
if os.path.exists(csv_path):
    selected_path = csv_path
elif os.path.exists(local_csv_path):
    selected_path = local_csv_path

if selected_path:
    print(f"📂 กำลังโหลดข้อมูลจากตำแหน่ง: {selected_path}")
    
    # 3. อ่านไฟล์ข้อมูลด้วย Pandas (รองรับการเข้ารหัสภาษาไทย tis-620)
    try:
        df = pd.read_csv(selected_path, encoding='tis-620')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(selected_path, encoding='utf-8')
        except:
            df = pd.read_csv(selected_path)
    
    # 4. แสดงขนาดข้อมูล
    print(f"📊 ขนาดชุดข้อมูล: พบพรรณไม้ทั้งหมด {df.shape[0]} รายการ | มีคอลัมน์ข้อมูลทั้งหมด {df.shape[1]} คอลัมน์\n")
    
    # 5. แสดงตารางแสดงข้อมูลพืชพรรณ
    print("--- ตารางแสดงข้อมูลพืชพรรณ ---")
    display(df.head(10))
    
    # 6. ส่งออกข้อมูลเป็นไฟล์ JSON (Export to JSON)
    # กำหนดตำแหน่งปลายทางสำหรับบันทึก JSON
    json_path = '/content/drive/MyDrive/Colab Notebooks/plants/data/VRU.json'
    local_json_path = 'data/VRU.json'
    
    # เลือกบันทึกตามสภาพแวดล้อมที่ใช้งานจริง
    target_json_path = json_path if selected_path == csv_path else local_json_path
    
    # บันทึกไฟล์เป็น JSON (orient='records', force_ascii=False เพื่อให้อ่านภาษาไทยใน JSON ได้ทันที)
    df.to_json(target_json_path, orient='records', force_ascii=False, indent=4)
    print(f"\n🎉 ส่งออกข้อมูลพรรณไม้เป็นไฟล์ JSON สำเร็จที่: {target_json_path}")
else:
    print(f"⚠️ ไม่พบไฟล์ CSV ทั้งบน Google Drive ({csv_path}) และในเครื่อง Local")

In [1]:
import pandas as pd
import numpy as np
import requests
import os
import json
import shutil
import random
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers

# 1. เชื่อมต่อกับ Google Drive (ยกเลิก Comment เมื่อรันบน Colab จริง)
from google.colab import drive
drive.mount('/content/drive')

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
TensorFlow Version: 2.20.0
GPU Available: []


In [7]:
# 1. กำหนดพาธไฟล์รายชื่อพืชพรรณและไฟล์ผลลัพธ์ลิงก์รูปภาพ
json_path = '/content/drive/MyDrive/Colab Notebooks/plants/data/VRU.json'
local_json_path = 'data/VRU.json'

# ตรวจสอบและเลือกพาธที่มีไฟล์อยู่จริง
selected_path = json_path if os.path.exists(json_path) else (local_json_path if os.path.exists(local_json_path) else None)

# 2. ฟังก์ชันส่งคำขอไปยัง GBIF API เพื่อรวบรวมลิงก์รูปภาพพืช
def fetch_gbif_image_links(dataframe, limit_per_species=100):
    print(f"🔍 เริ่มค้นหาภาพจาก GBIF (ดึงสูงสุด {limit_per_species} รูปภาพต่อพืช 1 ชนิด)...")
    image_records = []
    
    for index, row in dataframe.iterrows():
        scientific_name = row.get('scientific_name', '')
        common_name = row.get('common_name_th', 'Unknown_Plant')
        family = row.get('family', 'Unknown_Family')
        
        if not scientific_name or pd.isna(scientific_name):
            continue
            
        print(f"-> กำลังค้นพืช: {common_name} ({scientific_name}) ", end="")
        
        url = "https://api.gbif.org/v1/occurrence/search"
        params = {
            "scientificName": scientific_name,
            "mediaType": "StillImage",
            "basisOfRecord":"HUMAN_OBSERVATION",
            "limit": limit_per_species
        }
        
        try:
            response = requests.get(url, params=params, timeout=15)
            if response.status_code == 200:
                data = response.json()
                results = data.get("results", [])

                found_count = data.get("count")
                image_records.append({
                    "common_name_th": common_name,
                    "scientific_name": scientific_name,
                    "family": family,
                    "count": found_count
                })

                # found_count = 0
                # for record in results:
                #     media_list = record.get("media", [])
                #     for media in media_list:
                #         if media.get("type") == "StillImage":
                #             img_url = media.get("identifier")
                #             if img_url:
                #                 image_records.append({
                #                     "common_name_th": common_name,
                #                     "scientific_name": scientific_name,
                #                     "family": family,
                #                     "image_url": img_url
                #                 })
                #                 found_count += 1
                #                 break # ดึงเพียง 1 ลิงก์รูปภาพต่อ 1 Occurrence Record
                print(f"[พบภาพ {found_count} รูป]")
            else:
                print(f"[Error: Server Status {response.status_code}]")
        except Exception as e:
            print(f"[Error: {e}]")
            
    return pd.DataFrame(image_records)

# 3. เรียกใช้งาน (ปลดล็อกเมื่อรันจริงใน Google Colab)
if selected_path:
    df_plants = pd.read_json(selected_path)
    df_images = fetch_gbif_image_links(df_plants, limit_per_species=10)
    
    # ส่งออกลิงก์ภาพพืชที่ดึงได้ไปที่ไฟล์ VRU_image_links.json
    target_links_path = '/content/drive/MyDrive/Colab Notebooks/plants/data/VRU_image_meta.json' if selected_path == json_path else 'data/VRU_image_meta.json'
    df_images.to_json(target_links_path, orient='records', force_ascii=False, indent=4)
    print(f"\n🎉 บันทึกตารางลิงก์ภาพพืชเรียบร้อยที่: {target_links_path}")
    print(f"📂 พบไฟล์รายชื่อพืชที่: {selected_path} (ระบบดึงลิงก์จาก GBIF API พร้อมรัน)")
else:
    print("⚠️ ไม่พบไฟล์ฐานข้อมูลรายชื่อพืช VRU.json โปรดรัน data.ipynb เพื่อจัดเตรียมก่อน")

🔍 เริ่มค้นหาภาพจาก GBIF (ดึงสูงสุด 10 รูปภาพต่อพืช 1 ชนิด)...
-> กำลังค้นพืช: กรรณิกา (Nyctanthes arbor-tristis L.) [พบภาพ 466 รูป]
-> กำลังค้นพืช: กะตังใบ (Leea indica (Burm. f.) Merr.) [พบภาพ 288 รูป]
-> กำลังค้นพืช: กระถิน (Leucaena leucocephala (Lam.) de Wit) [พบภาพ 18971 รูป]
-> กำลังค้นพืช: กระถินณรงค์ (Acacia auriculiformis A. Cunn. ex Benth.) [พบภาพ 2510 รูป]
-> กำลังค้นพืช: กระดังงาจีน (Artabotrys hexapetalus (L. f.) Bhandari) [พบภาพ 84 รูป]
-> กำลังค้นพืช: กระดังงาสงขลา (Cananga odorata (Lam.) Hook. f. & Thomson var. fruticosa (Craib) J. Sinclair) [พบภาพ 9 รูป]
-> กำลังค้นพืช: กระท้อน (Sandoricum koetjape (Burm. f.) Merr.) [พบภาพ 91 รูป]
-> กำลังค้นพืช: กระทิง (Calophyllum inophyllum L.) [พบภาพ 2008 รูป]
-> กำลังค้นพืช: กระทุ่มนา (Mitragyna diversifolia (Wall. ex G. Don) Havil.) [พบภาพ 2 รูป]
-> กำลังค้นพืช: กระพี้จั่น (Millettia brandisiana Kurz) [พบภาพ 6 รูป]
-> กำลังค้นพืช: กันเกรา (Fagraea fragrans Roxb.) [พบภาพ 145 รูป]
-> กำลังค้นพืช: กร่าง (Ficus altissima Blume) [พบภา